## Imports

In [ ]:
#anaconda prompt > conda install -c anaconda openpyxl

import openpyxl

from bs4 import BeautifulSoup
import requests
import time
import datetime


import warnings
warnings.filterwarnings('ignore')

## Funcion para extraccion de datos Amazon

In [ ]:
# Conectar y obtener datos desde Amazon

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

#Funcion para obtener el precio de un producto
def get_amazon_price(URL):

    #Se obtiene el HTML del producto
    page = requests.get(URL, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find("span", { "class" : "a-price" }).find("span", recursive=False).text.strip()
        #price = soup1.find('span', {'class': 'a-offscreen'}).get_text() #if soup1.find('span', {'class': 'a-offscreen'})
    except AttributeError:
        price = "No se ha podido obtener el precio para este producto."
        
    return price

## Configuracion del Excel

In [ ]:
#Incluir la localizacion del fichero excel con los datos de Amazon:
path = r"C:\Users\jgmeras\Documents\Python Scripts\Precios.xlsx"

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row
column_limit = sheet_obj.max_column
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el ASIN (A=1, B=2, C=3...)
asin_col = 1

#Introducir el numero de la fila en la que se encuentra el primer valor de ASIN
first_asin_row = 2

# se comprueba mostrando por pantalla que la columna de ASIN se esta obteniendo correctamente 
# print("\nValores de ASIN a obtener:")
# for i in range(first_asin_row, row_limit + 1): 
#     cell_obj = sheet_obj.cell(row = i, column = asin_col) 
#     print(cell_obj.value)

In [ ]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
web_prefix = "https://www.amazon.es/dp/"

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 3

for i in range(first_asin_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = asin_col)
    ASIN = cell_obj.value
    URL = web_prefix+ASIN
    price = get_amazon_price(URL)
    print(ASIN + " tiene un precio de: " + price)
    
    #Se introduce el precio en el Excel
    c1 = sheet_obj.cell(row = i, column = precio_col)
    c1.value = price
    
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(r"C:\Users\jgmeras\Documents\Python Scripts\Precios.xlsx")
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")